# IEEE fraud model training: balancing test

Same training path as `02_model_training.ipynb`. The training set is first cut to `MAX_TRAIN_ROWS` with a stratified sample, so the fraud rate stays about 3.5%. Random oversampling then duplicates minority-class rows (`is_fraud == 1`) in that smaller training set and in the full validation set until both classes have the same count. The test set is left at the natural fraud rate.

A 1:1 oversample of the full training set would copy the 320k legitimate rows and land near 640k rows. Capping first keeps the balanced training set near twice `MAX_TRAIN_ROWS`.

The F1 threshold is chosen on the oversampled validation set. Test metrics are the comparison with the unbalanced run.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). That server proxies artifacts into the MinIO bucket `s3://mlflow/` (`http://localhost:9000`).

In [1]:
from pathlib import Path
import os

import mlflow
import pandas as pd

from src.data_processing.balancing_techniques import oversample_data
from src.run_time_configuration import BaseConfigParams
from src.model_training.training import ModelTrainingWorkflow

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
MAX_TRAIN_ROWS = 100_000
TOP_N_FEATURES = 40
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]

In [2]:
def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Processed dir: /home/martinezyamamotoarthur/modeling_fraud_system/data/processed
MLflow tracking URI: http://localhost:5000


In [3]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))

train (331633, 332) pos_rate 0.0352
validation (82909, 332) pos_rate 0.0352
test (175998, 332) pos_rate 0.0344


## Random oversampling

`RandomOverSampler` copies existing fraud rows. No synthetic samples. The training frame is capped first. Validation is balanced in full. Test is unchanged.

In [ ]:
from sklearn.model_selection import train_test_split


def class_counts(df: pd.DataFrame) -> pd.Series:
    return df[TARGET_COLUMN].value_counts().sort_index()


def limit_training_rows(df: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df
    kept, _ = train_test_split(
        df,
        train_size=max_rows,
        random_state=seed,
        stratify=df[TARGET_COLUMN],
    )
    return kept.reset_index(drop=True)


print("train loaded", df_train.shape)
print(class_counts(df_train))
df_train = limit_training_rows(df_train, MAX_TRAIN_ROWS, SEED)
print("train capped", df_train.shape)
print(class_counts(df_train))
df_train = oversample_data(df_train, TARGET_COLUMN, random_state=SEED)
print("train after", df_train.shape)
print(class_counts(df_train))

print("validation before")
print(class_counts(df_validation))
df_validation = oversample_data(df_validation, TARGET_COLUMN, random_state=SEED)
print("validation after", df_validation.shape)
print(class_counts(df_validation))

print("test", df_test.shape, "pos_rate", round(float(df_test[TARGET_COLUMN].mean()), 4))

train before
is_fraud
0    319953
1     11680
Name: count, dtype: int64


In [ ]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud",
    mlflow_run_name="ieee_xgb_oversample",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)

workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(numeric_only=True)

print("train", workflow.x_train_preproc.shape, "pos_rate", float(workflow.y_train.mean()))
print("valid", workflow.x_validation_preproc.shape, "pos_rate", float(workflow.y_validation.mean()))
print("test ", workflow.x_test_preproc.shape, "pos_rate", float(workflow.y_test.mean()))
print("numeric features", len(workflow.x_train_preproc.columns))

In [ ]:
results = workflow.train_model(
    hyperparameters={
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="ieee_xgb_oversample_01",
    top_n_features=TOP_N_FEATURES,
)

metrics_table = pd.DataFrame(
    {
        "train": results["train_metrics"],
        "validation": results["val_metrics"],
        "test": results["test_metrics"],
    }
).loc[["accuracy", "precision", "recall", "f1", "auc", "pr_auc", "ks_statistic"]]

run_id = workflow.run_time_config.mlflow_run_id
client = mlflow.tracking.MlflowClient()
run = client.get_run(run_id)

print("threshold", round(results["threshold"], 4))
print("mlflow run id", run_id)
print("tracking uri", mlflow.get_tracking_uri())
print("artifact uri", run.info.artifact_uri)
print("logged artifacts", [a.path for a in client.list_artifacts(run_id)])
display(results["top_features"])
metrics_table.round(4)